# API quickstart

A cell-by-cell version of the [script quickstart](../../examples/api/quickstart.py). The hand-written arrays demonstrate API usage. From the repository root, install `python -m pip install -e ".[notebook]"`.


In [10]:
from pathlib import Path

import numpy as np

from llm_scorekit import (
    automated_error_rate,
    automation_rate,
    average_set_size,
    calibrate,
    empirical_coverage,
    load_policy,
    save_policy,
)


## 1. `calibrate(...)`

**Pass:**

- `calibration_probabilities`: finite, nonnegative weights in a request-by-class
  matrix; each row sums to one.
- `calibration_labels`: the correct column's integer index for each row.
- `class_names`: names in that same column order.
- `method` and `alpha`: here LAC with an 80% coverage target (`alpha=0.20`).

**Returns:** a `CalibratedPolicy` holding the fitted threshold and fixed settings.


In [11]:
class_names = ("billing", "delivery", "returns")

# Each label identifies the correct column in the corresponding row.
calibration_probabilities = np.array(
    [
        [0.80, 0.10, 0.10],
        [0.10, 0.75, 0.15],
        [0.10, 0.20, 0.70],
        [0.65, 0.20, 0.15],
        [0.25, 0.60, 0.15],
        [0.20, 0.25, 0.55],
        [0.50, 0.30, 0.20],
        [0.30, 0.45, 0.25],
        [0.35, 0.25, 0.40],
        [0.35, 0.40, 0.25],
    ]
)
calibration_labels = np.array([0, 1, 2, 0, 1, 2, 0, 1, 2, 0])

policy = calibrate(
    calibration_probabilities,
    calibration_labels,
    class_names=class_names,
    method="lac",
    alpha=0.20,
)

policy


CalibratedPolicy(method='lac', alpha=0.2, threshold=0.6, class_names=('billing', 'delivery', 'returns'), regularization=None)

## 2. `save_policy(...)` / `load_policy(...)`

- `save_policy(policy, path)` receives a calibrated policy and a file path.
  It writes the JSON and returns `None`.
- `load_policy(path)` receives that path and returns a `CalibratedPolicy`,
  ready to predict without recalibrating.

Only settings are saved, not the model or data. Run from the repository root
or this notebook's folder; reruns replace the separate `notebook_policy.json`.


In [12]:
# Locate the repository when the kernel starts in this notebook's folder.
project_directory = Path.cwd()
if project_directory.name == "api" and project_directory.parent.name == "notebooks":
    project_directory = project_directory.parents[1]

relative_path = Path("outputs/quickstart/notebook_policy.json")
policy_path = project_directory / relative_path
policy_path.parent.mkdir(parents=True, exist_ok=True)

save_policy(policy, policy_path)
restored_policy = load_policy(policy_path)

print(f"Saved and reloaded: {relative_path}")


Saved and reloaded: outputs/quickstart/notebook_policy.json


## 3. `policy.predict(...)`

**Pass:** new normalized rows, using the same model, scoring procedure and
class-column order. True labels are not required.

**Returns:** a `DecisionBatch` with:

- `prediction_sets`: a boolean matrix matching the input shape; `True` retains a class.
- `accepted`: one boolean per request, true only for a one-label set.
- `predictions`: a list with one class-column index or `None` per request.
  Empty and multi-label sets are deferred.


In [6]:
evaluation_probabilities = np.array(
    [
        [0.80, 0.10, 0.10],
        [0.45, 0.45, 0.10],
        [0.34, 0.33, 0.33],
        [0.10, 0.75, 0.15],
        [0.15, 0.20, 0.65],
    ]
)

decisions = restored_policy.predict(evaluation_probabilities)
decisions


DecisionBatch(prediction_sets=array([[ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [False,  True, False],
       [False, False,  True]]), accepted=array([ True, False, False,  True,  True]), predictions=[0, None, None, 1, 2])

In [7]:
# Known answers are used for inspection and evaluation, not for routing.
evaluation_labels = np.array([0, 1, 2, 2, 2])

for sample_index, prediction in enumerate(decisions.predictions):
    included_indices = np.flatnonzero(decisions.prediction_sets[sample_index])
    included_names = [restored_policy.class_names[index] for index in included_indices]

    if prediction is None:
        decision = "defer"
    else:
        predicted_name = restored_policy.class_names[prediction]
        decision = f"accept {predicted_name}"

    true_class = evaluation_labels[sample_index]
    true_name = restored_policy.class_names[true_class]
    print(
        f"Request {sample_index + 1}: {included_names} -> {decision}; "
        f"true={true_name}"
    )


Request 1: ['billing'] -> accept billing; true=billing
Request 2: ['billing', 'delivery'] -> defer; true=delivery
Request 3: [] -> defer; true=returns
Request 4: ['delivery'] -> accept delivery; true=returns
Request 5: ['returns'] -> accept returns; true=returns


## 4. Evaluate the decisions

Pass the returned sets, acceptance mask and predictions to the existing metrics,
alongside true evaluation labels where needed. They return numerical rates or
an average set size.

Coverage counts all requests; automated-case error counts only accepted ones
and is `NaN` if none are accepted.


In [8]:
coverage = empirical_coverage(decisions.prediction_sets, evaluation_labels)
automated_fraction = automation_rate(decisions.accepted)
automated_error = automated_error_rate(
    decisions.predictions,
    evaluation_labels,
    decisions.accepted,
)
mean_set_size = average_set_size(decisions.prediction_sets)

print(f"Coverage: {coverage:.1%}")
print(f"Automation: {automated_fraction:.1%}")
print(f"Automated-case error: {automated_error:.1%}")
print(f"Average set size: {mean_set_size:.2f}")


Coverage: 60.0%
Automation: 60.0%
Automated-case error: 33.3%
Average set size: 1.00


## Use your own data

Replace these arrays with your model's normalized class weights and labels.
Keep calibration separate from training, tuning and evaluation. Set coverage
is not a guarantee on automatic-routing errors.

See the [usage guide](../../docs/usage.md) for input requirements and other methods.
